# Colab runner
Run cells 1-6 at the start of EVERY session (runtime must be GPU: Runtime > Change runtime type).

**Before first run:** add `GITHUB_TOKEN` and `HF_TOKEN` in the Secrets (key icon) and set the two variables in cell 2.

In [ ]:
# 1. Mount Drive (persistent storage for checkpoints/data)
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/clip_project'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.environ['PROJECT_DIR'] = PROJECT_DIR

In [ ]:
# 2. Clone or update the repo
from google.colab import userdata
GH_USER = 'YOUR_GITHUB_USERNAME'
GH_REPO = 'YOUR_REPO_NAME'
BRANCH  = 'main'   # or your own branch
token = userdata.get('GITHUB_TOKEN')
if not os.path.exists(f'/content/{GH_REPO}'):
    !git clone https://{token}@github.com/{GH_USER}/{GH_REPO}.git /content/{GH_REPO}
%cd /content/{GH_REPO}
!git checkout {BRANCH}
!git pull
!git config user.name  "YOUR NAME"
!git config user.email "you@example.com"

In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q

In [ ]:
# 4. Hugging Face login (Winoground is gated: accept its terms on the dataset page first)
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
# 5. GPU check
!nvidia-smi

In [ ]:
# 6. Copy preprocessed data from Drive to local disk (much faster than reading from Drive)
# One-time: put coco_subset.zip and negatives.jsonl in PROJECT_DIR/data/
os.makedirs('data', exist_ok=True)
!cp {PROJECT_DIR}/data/negatives.jsonl data/ 2>/dev/null || echo 'negatives.jsonl not on Drive yet'
if os.path.exists(f'{PROJECT_DIR}/data/coco_subset.zip') and not os.path.exists('data/coco_subset'):
    !unzip -q {PROJECT_DIR}/data/coco_subset.zip -d data/
!ls data

## Day 1 check: zero-shot Winoground baseline

In [ ]:
!python -m src.eval_winoground --out {PROJECT_DIR}/results/zeroshot_winoground.json

## Training (resumable: re-run the same cell after a disconnect)

In [ ]:
!python -m src.train --config configs/full_all_negs.yaml --seed 0 --out_dir {PROJECT_DIR}

## Run everything (skips finished runs)

In [ ]:
!python scripts/run_all.py

## Save small results back to GitHub (only if you want to push from Colab)

In [ ]:
!cp {PROJECT_DIR}/results/*.json results/ 2>/dev/null
!git add results/*.json
!git commit -m "add results" 
!git push